In [1]:
from pathlib import Path
from datasets import load_from_disk
import pandas as pd
import numpy as np
import time
import re

SEED = 42

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

dataset = load_from_disk(
    str(project_dir / "data" / "healthbrief_sampled")
)

validation = dataset["validation"]

rng = np.random.default_rng(SEED)

selected_ids = sorted(
    rng.choice(
        len(validation),
        size=20,
        replace=False
    ).tolist()
)

eval_df = validation.select(selected_ids).to_pandas()
eval_df.insert(0, "sample_id", selected_ids)

def lead3_summarize(text):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    return " ".join(sentences[:3])

lead3_summaries = []
lead3_times = []

for article in eval_df["article"]:
    start = time.perf_counter()

    summary = lead3_summarize(article)

    elapsed = time.perf_counter() - start

    lead3_summaries.append(summary)
    lead3_times.append(elapsed)

eval_df["lead3_summary"] = lead3_summaries
eval_df["lead3_time"] = lead3_times

print("Selected samples:", len(eval_df))
print("Selected IDs:", selected_ids)
print("Average Lead-3 inference time:", np.mean(lead3_times))

print("\nREFERENCE SUMMARY:\n")
print(eval_df.iloc[0]["summary"][:500])

print("\nLEAD-3 SUMMARY:\n")
print(eval_df.iloc[0]["lead3_summary"][:500])

Selected samples: 20
Selected IDs: [7, 8, 12, 17, 36, 45, 47, 49, 54, 61, 63, 68, 71, 73, 75, 83, 84, 86, 89, 94]
Average Lead-3 inference time: 0.0004137184499995783

REFERENCE SUMMARY:

Genetic material is contained within molecules of DNA . In plants and many other organisms , these DNA molecules are packaged around proteins called histones to make a structure known as chromatin . Altering how the DNA is packaged in chromatin can control the activity of genes . For example , a group of proteins called the Polycomb Repressive Complex 2 ( PRC2 ) adds methyl tags to histones , which can alter the packaging of chromatin to lower the activity of particular genes . When a cell divid

LEAD-3 SUMMARY:

Inheritance of gene expression states is fundamental for cells to ‘remember’ past events , such as environmental or developmental cues . The conserved Polycomb Repressive Complex 2 ( PRC2 ) maintains epigenetic repression of many genes in animals and plants and modifies chromatin at its targe

/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

t5_summaries = []
t5_times = []

for article in eval_df["article"]:
    prompt = (
        "Summarize the following biomedical research article "
        "in plain language for a general audience:\n\n"
        + article
    )

    start = time.perf_counter()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            num_beams=1,
            do_sample=False
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    summary = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    elapsed = time.perf_counter() - start

    t5_summaries.append(summary)
    t5_times.append(elapsed)

eval_df["t5_summary"] = t5_summaries
eval_df["t5_time"] = t5_times

print("Device:", device)
print("Generated summaries:", len(t5_summaries))
print("Average inference time:", round(np.mean(t5_times), 4))

print("\nREFERENCE SUMMARY:\n")
print(eval_df.iloc[0]["summary"][:500])

print("\nFLAN-T5 SUMMARY:\n")
print(eval_df.iloc[0]["t5_summary"][:500])

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 14737.54it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
In file included from /home/pasali/anaconda3/envs/myenv/include/python3.14/Python.h:14,
                 from /home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/triton/backends/nvidia/driver.c:9:
/home/pasali/anaconda3/envs/myenv/include/python3.14/pyconfig.h:2007:9: warning: ‘_POSIX_C_SOURCE’ redefined
 2007 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/x86_64-linux-gnu/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/triton/backends/nvidia/include/cuda.

Device: cuda
Generated summaries: 20
Average inference time: 0.3837

REFERENCE SUMMARY:

Genetic material is contained within molecules of DNA . In plants and many other organisms , these DNA molecules are packaged around proteins called histones to make a structure known as chromatin . Altering how the DNA is packaged in chromatin can control the activity of genes . For example , a group of proteins called the Polycomb Repressive Complex 2 ( PRC2 ) adds methyl tags to histones , which can alter the packaging of chromatin to lower the activity of particular genes . When a cell divid

FLAN-T5 SUMMARY:

Inheritance of gene expression states is fundamental for cells to 'remember' past events , such as environmental or developmental cues .


In [3]:
%pip install -q rouge-score

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

models = {
    "Lead-3": ("lead3_summary", "lead3_time"),
    "FLAN-T5-small": ("t5_summary", "t5_time")
}

results = []

for model_name, (summary_col, time_col) in models.items():
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for reference, prediction in zip(
        eval_df["summary"],
        eval_df[summary_col]
    ):
        scores = scorer.score(reference, prediction)

        rouge1_scores.append(scores["rouge1"].fmeasure)
        rouge2_scores.append(scores["rouge2"].fmeasure)
        rougeL_scores.append(scores["rougeL"].fmeasure)

    results.append({
        "Model": model_name,
        "ROUGE-1": round(np.mean(rouge1_scores) * 100, 2),
        "ROUGE-2": round(np.mean(rouge2_scores) * 100, 2),
        "ROUGE-L": round(np.mean(rougeL_scores) * 100, 2),
        "Inference Time (s)": round(eval_df[time_col].mean(), 4)
    })

comparison_df = pd.DataFrame(results)

reports_dir = project_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    reports_dir / "baseline_comparison.csv",
    index=False
)

display(comparison_df)

print("\nSaved:", reports_dir / "baseline_comparison.csv")

Note: you may need to restart the kernel to use updated packages.


,Model,ROUGE-1,ROUGE-2,ROUGE-L,Inference Time (s)
0,Lead-3,17.61,4.72,10.52,0.0004
1,FLAN-T5-small,4.33,1.14,3.47,0.3837



Saved: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/reports/baseline_comparison.csv


In [4]:
review_df = eval_df.head(5).copy()

for _, row in review_df.iterrows():
    print("=" * 80)
    print("SAMPLE ID:", row["sample_id"])

    print("\nFLAN-T5 SUMMARY:\n")
    print(row["t5_summary"])

    print("\nREFERENCE SUMMARY:\n")
    print(row["summary"])

    print("\nSOURCE ARTICLE - BEGINNING:\n")
    print(row["article"][:1200])

    print("\nSOURCE ARTICLE - END:\n")
    print(row["article"][-1500:])

    print("\n")


def find_source_evidence(sample_id, phrase, window=300):
    article = review_df.loc[
        review_df["sample_id"] == sample_id,
        "article"
    ].iloc[0]

    matches = list(
        re.finditer(re.escape(phrase), article, flags=re.IGNORECASE)
    )

    if not matches:
        print("No exact phrase match found.")
        return

    for match in matches[:5]:
        start = max(0, match.start() - window)
        end = min(len(article), match.end() + window)

        print(article[start:end])
        print("-" * 60)

SAMPLE ID: 7

FLAN-T5 SUMMARY:

Inheritance of gene expression states is fundamental for cells to 'remember' past events , such as environmental or developmental cues .

REFERENCE SUMMARY:

Genetic material is contained within molecules of DNA . In plants and many other organisms , these DNA molecules are packaged around proteins called histones to make a structure known as chromatin . Altering how the DNA is packaged in chromatin can control the activity of genes . For example , a group of proteins called the Polycomb Repressive Complex 2 ( PRC2 ) adds methyl tags to histones , which can alter the packaging of chromatin to lower the activity of particular genes . When a cell divides , it is sometimes important that genes in the daughter cells have similar levels of activity as the parent cell . This allows individual cells to ‘remember’ past events , such as exposure to cold temperatures or other environmental conditions . The pattern of methyl tags on histones can be passed onto the 

In [5]:
manual_analysis = [
    {
        "sample_id": 7,
        "supported_claims": "Gene expression states can preserve cellular memory.",
        "contradicted_claims": "",
        "unverifiable_claims": "",
        "source_evidence": "Inheritance of gene expression states is fundamental for cells to remember past events.",
        "unsupported_information": 0,
        "incorrect_numeric": 0,
        "exaggerated_result": 0,
        "important_omission": 1,
        "uncertainty_to_certainty": 0,
        "notes": "FLC gene experiments and cis epigenetic memory findings are omitted."
    },
    {
        "sample_id": 8,
        "supported_claims": "AAA+ chaperones use ATP-dependent mechanisms.",
        "contradicted_claims": "Activation by coiled-coil membranes.",
        "unverifiable_claims": "",
        "source_evidence": "ClpC activation involves coiled-coil middle domains and MecA.",
        "unsupported_information": 1,
        "incorrect_numeric": 0,
        "exaggerated_result": 0,
        "important_omission": 1,
        "uncertainty_to_certainty": 0,
        "notes": "Incorrect biological terminology and missing regulatory mechanism."
    },
    {
        "sample_id": 12,
        "supported_claims": "",
        "contradicted_claims": "TBP2 is required for muscle-gene expression.",
        "unverifiable_claims": "",
        "source_evidence": "TBP2 does not replace TBP during muscle differentiation.",
        "unsupported_information": 0,
        "incorrect_numeric": 0,
        "exaggerated_result": 0,
        "important_omission": 1,
        "uncertainty_to_certainty": 1,
        "notes": "Previous scientific hypothesis is incorrectly presented as a confirmed finding."
    },
    {
        "sample_id": 17,
        "supported_claims": "SNr neurons can fire without synaptic input.",
        "contradicted_claims": "",
        "unverifiable_claims": "",
        "source_evidence": "Some neurons fire action potentials without synaptic input.",
        "unsupported_information": 0,
        "incorrect_numeric": 0,
        "exaggerated_result": 0,
        "important_omission": 1,
        "uncertainty_to_certainty": 0,
        "notes": "NALCN channel findings and experimental results are omitted."
    },
    {
        "sample_id": 36,
        "supported_claims": "Forensic entomology is used to estimate postmortem intervals.",
        "contradicted_claims": "",
        "unverifiable_claims": "",
        "source_evidence": "The study investigates microbial community changes as a method for estimating postmortem intervals.",
        "unsupported_information": 0,
        "incorrect_numeric": 0,
        "exaggerated_result": 0,
        "important_omission": 1,
        "uncertainty_to_certainty": 0,
        "notes": "Misleading research focus. Microbial changes and the approximately three-day estimation accuracy are omitted."
    }
]

analysis_df = pd.DataFrame(manual_analysis)

analysis_df["generated_summary"] = analysis_df["sample_id"].map(
    review_df.set_index("sample_id")["t5_summary"]
)

analysis_df["review_scope"] = "Source excerpts and reference summary"

output_path = reports_dir / "baseline_error_analysis.csv"

analysis_df.to_csv(output_path, index=False)

error_columns = [
    "unsupported_information",
    "incorrect_numeric",
    "exaggerated_result",
    "important_omission",
    "uncertainty_to_certainty"
]

error_counts = analysis_df[error_columns].sum()

display(analysis_df)
print("\nERROR COUNTS")
display(error_counts)

print("\nSaved:", output_path)

,sample_id,supported_claims,contradicted_claims,unverifiable_claims,source_evidence,unsupported_information,incorrect_numeric,exaggerated_result,important_omission,uncertainty_to_certainty,notes,generated_summary,review_scope
0,7,Gene expression states can preserve cellular m...,,,Inheritance of gene expression states is funda...,0,0,0,1,0,FLC gene experiments and cis epigenetic memory...,Inheritance of gene expression states is funda...,Source excerpts and reference summary
1,8,AAA+ chaperones use ATP-dependent mechanisms.,Activation by coiled-coil membranes.,,ClpC activation involves coiled-coil middle do...,1,0,0,1,0,Incorrect biological terminology and missing r...,Activation of ATP-fueled chaperones by coiled-...,Source excerpts and reference summary
2,12,,TBP2 is required for muscle-gene expression.,,TBP2 does not replace TBP during muscle differ...,0,0,0,1,1,Previous scientific hypothesis is incorrectly ...,TBP2 is required for activation of muscle-gene...,Source excerpts and reference summary
3,17,SNr neurons can fire without synaptic input.,,,Some neurons fire action potentials without sy...,0,0,0,1,0,NALCN channel findings and experimental result...,SNr neurons are capable of firing action poten...,Source excerpts and reference summary
4,36,Forensic entomology is used to estimate postmo...,,,The study investigates microbial community cha...,0,0,0,1,0,Misleading research focus. Microbial changes a...,forensic entomology method for estimating the ...,Source excerpts and reference summary



ERROR COUNTS


unsupported_information     1
incorrect_numeric           0
exaggerated_result          0
important_omission          5
uncertainty_to_certainty    1
dtype: int64


Saved: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/reports/baseline_error_analysis.csv


In [10]:
from pathlib import Path
from datasets import load_from_disk
import pandas as pd
import numpy as np
import re

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

reports_dir = project_dir / "reports"

dataset = load_from_disk(
    str(project_dir / "data" / "healthbrief_sampled")
)

comparison_df = pd.read_csv(
    reports_dir / "baseline_comparison.csv"
)

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text.strip()).casefold()

def valid_text(text):
    return isinstance(text, str) and bool(text.strip())


split_sizes = {
    "train": 500,
    "validation": 100,
    "test": 100
}

split_integrity = all(
    split in dataset and len(dataset[split]) == size
    for split, size in split_sizes.items()
)


article_sets = {
    split: {
        normalize_text(text)
        for text in dataset[split]["article"]
        if valid_text(text)
    }
    for split in split_sizes
}

overlaps = {
    "train_validation": len(
        article_sets["train"] & article_sets["validation"]
    ),
    "train_test": len(
        article_sets["train"] & article_sets["test"]
    ),
    "validation_test": len(
        article_sets["validation"] & article_sets["test"]
    )
}

data_leakage = all(
    count == 0 for count in overlaps.values()
)


valid_articles = all(
    all(valid_text(text) for text in dataset[split]["article"])
    for split in split_sizes
)

valid_summaries = all(
    all(valid_text(text) for text in dataset[split]["summary"])
    for split in ["train", "validation"]
)

valid_texts = valid_articles and valid_summaries


prompts = [
    "Summarize the following biomedical research article "
    "in plain language for a general audience:\n\n" + article
    for article in eval_df["article"]
]

encoded = tokenizer(
    prompts,
    truncation=True,
    max_length=512,
    padding=False
)

input_lengths = [
    len(ids) for ids in encoded["input_ids"]
]

input_length_valid = (
    len(input_lengths) == 20
    and all(0 < length <= 512 for length in input_lengths)
)


expected_columns = [
    "sample_id",
    "lead3_summary",
    "t5_summary"
]

columns_valid = all(
    column in eval_df.columns
    for column in expected_columns
)

output_diagnostics = []
invalid_outputs = []

if columns_valid:
    for column in ["lead3_summary", "t5_summary"]:
        valid_mask = eval_df[column].apply(valid_text)

        output_diagnostics.append({
            "model": column,
            "total": len(eval_df),
            "valid": int(valid_mask.sum()),
            "invalid": int((~valid_mask).sum())
        })

        if not valid_mask.all():
            invalid_rows = eval_df.loc[
                ~valid_mask,
                ["sample_id", column]
            ]

            for _, row in invalid_rows.iterrows():
                invalid_outputs.append({
                    "model": column,
                    "sample_id": row["sample_id"],
                    "generated_text": repr(row[column])
                })

    output_validity = (
        len(eval_df) == 20
        and eval_df["sample_id"].notna().all()
        and eval_df["sample_id"].nunique() == 20
        and all(
            result["valid"] == 20
            for result in output_diagnostics
        )
    )
else:
    output_validity = False


metric_columns = [
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "Inference Time (s)"
]

expected_models = {
    "Lead-3",
    "FLAN-T5-small"
}

metric_structure_valid = (
    len(comparison_df) == 2
    and "Model" in comparison_df.columns
    and set(comparison_df["Model"]) == expected_models
    and all(
        column in comparison_df.columns
        for column in metric_columns
    )
)

if metric_structure_valid:
    numeric_metrics = comparison_df[metric_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    metric_validity = bool(
        np.isfinite(numeric_metrics.to_numpy()).all()
        and numeric_metrics[
            ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
        ].apply(
            lambda col: col.between(0, 100).all()
        ).all()
        and (
            numeric_metrics["Inference Time (s)"] >= 0
        ).all()
    )
else:
    metric_validity = False


test_results = {
    "Split Integrity": split_integrity,
    "Data Leakage": data_leakage,
    "Valid Text": valid_texts,
    "Input Length": input_length_valid,
    "Output Validity": output_validity,
    "Metric Validity": metric_validity
}

test_df = pd.DataFrame([
    {
        "Test": name,
        "Status": "PASS" if bool(result) else "FAIL"
    }
    for name, result in test_results.items()
])


print("=" * 60)
print("AUTOMATED TEST RESULTS")
print("=" * 60)

display(test_df)


print("\nDATA LEAKAGE DIAGNOSTICS")

for name, count in overlaps.items():
    print(f"{name}: {count}")


print("\nINPUT LENGTH DIAGNOSTICS")

print("Total inputs:", len(input_lengths))
print("Maximum length:", max(input_lengths))
print("Minimum length:", min(input_lengths))


print("\nOUTPUT VALIDITY DIAGNOSTICS")

if columns_valid:
    diagnostics_df = pd.DataFrame(output_diagnostics)
    display(diagnostics_df)

    if invalid_outputs:
        print("\nINVALID GENERATED SUMMARIES")
        display(pd.DataFrame(invalid_outputs))
    else:
        print("All generated summaries are valid.")
else:
    print("Required columns are missing.")


print("\nMETRIC VALIDITY DIAGNOSTICS")

display(comparison_df)


passed = sum(
    bool(result) for result in test_results.values()
)

total = len(test_results)

failed_tests = [
    name
    for name, result in test_results.items()
    if not result
]

print("\n" + "=" * 60)
print("FINAL TEST SUMMARY")
print("=" * 60)

print(f"Passed: {passed}/{total}")
print(f"Failed: {len(failed_tests)}")

if failed_tests:
    print("\nFAILED TESTS")

    for test in failed_tests:
        print(f"- {test}")
else:
    print("\nALL TESTS PASSED")


test_df.to_csv(
    reports_dir / "automated_test_results.csv",
    index=False
)

print("\nTest results saved successfully.")

AUTOMATED TEST RESULTS


,Test,Status
0,Split Integrity,PASS
1,Data Leakage,PASS
2,Valid Text,PASS
3,Input Length,PASS
4,Output Validity,FAIL
5,Metric Validity,PASS



DATA LEAKAGE DIAGNOSTICS
train_validation: 0
train_test: 0
validation_test: 0

INPUT LENGTH DIAGNOSTICS
Total inputs: 20
Maximum length: 512
Minimum length: 512

OUTPUT VALIDITY DIAGNOSTICS


,model,total,valid,invalid
0,lead3_summary,20,20,0
1,t5_summary,20,19,1



INVALID GENERATED SUMMARIES


,model,sample_id,generated_text
0,t5_summary,54,' ...



METRIC VALIDITY DIAGNOSTICS


,Model,ROUGE-1,ROUGE-2,ROUGE-L,Inference Time (s)
0,Lead-3,17.61,4.72,10.52,0.0004
1,FLAN-T5-small,4.33,1.14,3.47,0.3837



FINAL TEST SUMMARY
Passed: 5/6
Failed: 1

FAILED TESTS
- Output Validity

Test results saved successfully.
